[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/055.%20The%20Newsvendor%20Problem/P55-Tier-10_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 55. The Newsvendor Problem

## Tier 10: Proactive Demand and Market Shaping

This notebook extends the inventory-planning setting with decisions that influence demand. The example considers Marina's ski-jacket business: the company chooses a selling price, marketing spend, partnership intensity, and order quantity. The goal is to maximize expected profit under the model described below.

### Learning objectives

- Translate price, marketing, and partnership decisions into a demand estimate.
- Connect demand to sales, leftover inventory, and profit.
- Use differential evolution to search for a high-profit strategy over continuous decision ranges.
- Examine how the optimized strategy responds to changes in the seasonal demand factor.

### Scope and modeling assumptions

This is a single-period, deterministic demand-shaping model with an order quantity and a salvage value for unsold units. Demand is calculated from the selected decisions rather than represented by a probability distribution. Consequently, this notebook does not model demand uncertainty, multiple planning periods, customer segments, competitor responses, or the long-term effects of marketing and partnerships. Partnership intensity is treated as a continuous decision between 0 and 3, not as a count of discrete contracts. The model's results are conditional on these assumptions and should not be interpreted as a complete forecast of market behavior.

## Model and solution method

The decisions are collected in the vector $x=(p,m,r,Q)$: selling price $p$, marketing spend $m$, partnership intensity $r$, and order quantity $Q$. The model first estimates demand, then calculates sales and leftovers, and finally evaluates profit. This sequence connects the demand-shaping decisions to the inventory decision.

The modeled demand is

$$D(p,m,r;s)=\max\left\{100,\;D_0\left(\frac{p}{p_0}\right)^\epsilon\left[1+\alpha\ln\left(1+\frac{m}{1000}\right)\right](1+0.5r)s\right\}.$$

Here, $D_0$ is base demand, $p_0$ is the base price, $\epsilon$ is the price-elasticity exponent, $\alpha$ is marketing effectiveness, and $s$ is the seasonal factor. The lower bound of 100 units is a modeling assumption. Because demand is proportional to $p^\epsilon$ when the other factors are fixed, $\epsilon$ is the demand elasticity with respect to price in this power-law model. Marketing has a logarithmic response, so its modeled demand effect grows at a decreasing rate. Partnership intensity multiplies demand by $1+0.5r$.

For a given demand estimate, sales cannot exceed either demand or the order quantity. Any units ordered but not sold are leftovers:

$$S=\min(Q,D),\qquad L=\max(0,Q-D).$$

The profit objective is

$$\begin{aligned}
\Pi(p,m,r,Q;s)={}&pS-180Q-m-10000r+90L,\\
\max_{p,m,r,Q}\quad&\Pi(p,m,r,Q;s),\\
\text{subject to}\quad&200\leq p\leq 400,\\
&0\leq m\leq 50000,\\
&0\leq r\leq 3,\\
&500\leq Q\leq 2000.
\end{aligned}$$

The terms represent sales revenue, production cost, marketing cost, partnership cost, and salvage revenue, respectively. All monetary values are in dollars; quantities are in units. Lost sales carry no additional penalty in this model beyond the revenue not earned. The code minimizes $-\Pi$ because `scipy.optimize.differential_evolution` is a minimizer.

Differential evolution is a population-based search over the bounded, continuous decision space. It evaluates candidate decision vectors and iteratively updates the population; its result is a numerical search result, not a symbolic proof of global optimality. The fixed seed and search settings make the search reproducible under the same software environment. The principal implementation is `demand_function` for $D$, `profit_function` for $-\Pi$, and `optimize_strategy` for the bounded search.

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution

print("Libraries imported for demand shaping")

In [ ]:
@dataclass
class MarketParameters:
    """Parameters that define the demand response in the market model."""

    base_price: float = 320.0
    base_demand: float = 1000.0
    price_elasticity: float = -1.5
    marketing_effectiveness: float = 0.8
    partnership_discount: float = 0.15
    seasonal_factor: float = 1.0


market_params = MarketParameters()
print("Market parameters initialized")

In [ ]:
class DemandShapingOptimizer:
    def __init__(self, params):
        self.params = params

    def demand_function(self, price, marketing_spend, partnership_level):
        """Estimate deterministic demand for a given set of market decisions."""
        price_effect = (
            price / self.params.base_price
        ) ** self.params.price_elasticity

        # The logarithmic response represents diminishing demand gains from spend.
        marketing_effect = 1 + self.params.marketing_effectiveness * np.log(
            1 + marketing_spend / 1000
        )
        partnership_effect = 1 + 0.5 * partnership_level

        demand = (
            self.params.base_demand
            * price_effect
            * marketing_effect
            * partnership_effect
            * self.params.seasonal_factor
        )

        return max(100.0, demand)

    def profit_function(self, decisions):
        """Return negative profit for differential evolution to minimize.

        The decision vector is [price, marketing_spend,
        partnership_level, order_quantity].
        """
        price, marketing_spend, partnership_level, order_quantity = decisions

        demand = self.demand_function(
            price, marketing_spend, partnership_level
        )
        sales = min(order_quantity, demand)
        leftovers = max(0.0, order_quantity - demand)

        revenue = price * sales
        production_cost = 180 * order_quantity
        marketing_cost = marketing_spend
        partnership_cost = 10000 * partnership_level
        salvage_value = 90 * leftovers

        profit = (
            revenue
            - production_cost
            - marketing_cost
            - partnership_cost
            + salvage_value
        )
        return -profit

    def optimize_strategy(self):
        """Search the bounded decision space for a high-profit strategy."""
        bounds = [
            (200, 400),
            (0, 50000),
            (0, 3),
            (500, 2000),
        ]

        result = differential_evolution(
            self.profit_function,
            bounds,
            maxiter=100,
            popsize=15,
            seed=42,
        )

        optimal_price, optimal_marketing, optimal_partnership, optimal_order = (
            result.x
        )
        optimal_demand = self.demand_function(
            optimal_price, optimal_marketing, optimal_partnership
        )
        optimal_sales = min(optimal_order, optimal_demand)
        optimal_leftovers = max(0.0, optimal_order - optimal_demand)
        optimal_profit = -result.fun

        # This check links the reported objective value to the returned decisions.
        assert np.isclose(
            optimal_profit,
            -self.profit_function(result.x),
            rtol=1e-9,
            atol=1e-7,
        )

        return {
            "price": optimal_price,
            "marketing_spend": optimal_marketing,
            "partnership_level": optimal_partnership,
            "order_quantity": optimal_order,
            "demand": optimal_demand,
            "sales": optimal_sales,
            "leftovers": optimal_leftovers,
            "profit": optimal_profit,
        }


print("Demand-shaping optimizer defined")

## Optimize and compare with a reference policy

The optimizer chooses all four decisions together because each affects profit directly or through demand and inventory. The reference policy uses the base price and demand, with the stated order quantity from Tier 1; it has no marketing or partnership spend. It provides a point of comparison, not a second optimization.

In [ ]:
optimizer = DemandShapingOptimizer(market_params)
results = optimizer.optimize_strategy()

print("=== DEMAND-SHAPING OPTIMIZATION RESULTS ===")
print("\nOptimized decisions:")
print(f"  Price: ${results['price']:.2f}")
print(f"  Marketing spend: ${results['marketing_spend']:,.2f}")
print(f"  Partnership intensity: {results['partnership_level']:.3f}")
print(f"  Order quantity: {results['order_quantity']:.1f} units")
print("\nModeled outcomes:")
print(f"  Demand: {results['demand']:.1f} units")
print(f"  Sales: {results['sales']:.1f} units")
print(f"  Leftovers: {results['leftovers']:.1f} units")
print(f"  Profit: ${results['profit']:,.2f}")

traditional_price = 320.0
traditional_demand = 1000.0
traditional_order = 1011.0
traditional_profit = (
    traditional_price * min(traditional_order, traditional_demand)
    - 180 * traditional_order
    + 90 * max(0, traditional_order - traditional_demand)
)

print("\nReference policy:")
print(f"  Price: ${traditional_price:.2f}")
print(f"  Demand: {traditional_demand:.1f} units")
print(f"  Order quantity: {traditional_order:.1f} units")
print(f"  Profit: ${traditional_profit:,.2f}")

improvement = (
    (results["profit"] - traditional_profit) / traditional_profit * 100
)
print(f"\nProfit difference relative to the reference: {improvement:+.2f}%")

## Seasonal sensitivity analysis

Sensitivity analysis asks how the model's recommended decisions and outcomes change when an input assumption changes. Here, each scenario changes only the seasonal factor, then reruns the same optimizer with the same bounds and search settings. The original factor is restored after the experiment so that later cells use the baseline parameter setting.

In [ ]:
scenarios = [
    {"name": "Low Season", "seasonal_factor": 0.7},
    {"name": "Normal Season", "seasonal_factor": 1.0},
    {"name": "High Season", "seasonal_factor": 1.3},
    {"name": "Peak Season", "seasonal_factor": 1.5},
]

scenario_results = []
original_seasonal_factor = optimizer.params.seasonal_factor

try:
    for scenario in scenarios:
        optimizer.params.seasonal_factor = scenario["seasonal_factor"]
        scenario_result = optimizer.optimize_strategy()

        scenario_results.append(
            {
                "scenario": scenario["name"],
                "seasonal_factor": scenario["seasonal_factor"],
                "price": scenario_result["price"],
                "demand": scenario_result["demand"],
                "profit": scenario_result["profit"],
            }
        )
finally:
    optimizer.params.seasonal_factor = original_seasonal_factor

print("=== SEASONAL SENSITIVITY ANALYSIS ===")
print(
    f"{'Scenario':<15} {'Season':>8} {'Price':>10} "
    f"{'Demand':>12} {'Profit':>15}"
)
print("-" * 64)

for scenario_result in scenario_results:
    print(
        f"{scenario_result['scenario']:<15} "
        f"{scenario_result['seasonal_factor']:>8.1f} "
        f"${scenario_result['price']:>9.2f} "
        f"{scenario_result['demand']:>11.1f} "
        f"${scenario_result['profit']:>14,.2f}"
    )

## Visualize the optimized strategy and scenarios

The first panel shows how modeled demand varies with price while the other optimized decisions are held fixed. The second panel varies marketing spend while keeping the optimized price, partnership intensity, and order quantity fixed; its profit curve uses the same profit function as the optimizer. The seasonal panel compares each scenario's optimized profit and demand. These are model-based comparisons, not observations from a real market.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Proactive Demand and Market-Shaping Analysis", fontsize=16)

# Price changes demand through the elasticity term in the demand model.
ax1 = axes[0, 0]
price_range = np.linspace(200, 400, 50)
price_demands = [
    optimizer.demand_function(
        price,
        results["marketing_spend"],
        results["partnership_level"],
    )
    for price in price_range
]

ax1.plot(price_range, price_demands, color="blue", linewidth=2, label="Modeled demand")
ax1.axvline(
    results["price"],
    color="red",
    linestyle="--",
    label=f"Optimized price: ${results['price']:.2f}",
)
ax1.axhline(results["demand"], color="gray", linestyle=":", label="Demand at optimized price")
ax1.set_xlabel("Price (USD per unit)")
ax1.set_ylabel("Demand (units)")
ax1.set_title("Price and Modeled Demand")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Re-evaluating the shared objective keeps this curve consistent with the model.
ax2 = axes[0, 1]
marketing_range = np.linspace(0, 50000, 50)
marketing_demands = [
    optimizer.demand_function(
        results["price"],
        marketing_spend,
        results["partnership_level"],
    )
    for marketing_spend in marketing_range
]
marketing_profits = [
    -optimizer.profit_function(
        [
            results["price"],
            marketing_spend,
            results["partnership_level"],
            results["order_quantity"],
        ]
    )
    for marketing_spend in marketing_range
]

ax2_twin = ax2.twinx()
demand_line = ax2.plot(
    marketing_range,
    marketing_demands,
    color="blue",
    label="Demand (units)",
)[0]
profit_line = ax2_twin.plot(
    marketing_range,
    marketing_profits,
    color="red",
    label="Profit (USD)",
)[0]
ax2.axvline(
    results["marketing_spend"],
    color="green",
    linestyle="--",
    label="Optimized marketing spend",
)
ax2.set_xlabel("Marketing spend (USD)")
ax2.set_ylabel("Demand (units)", color="blue")
ax2_twin.set_ylabel("Profit (USD)", color="red")
ax2.set_title("Marketing Spend, Demand, and Profit")
ax2.legend(handles=[demand_line, profit_line], loc="upper left")
ax2.grid(True, alpha=0.3)

# The paired bars use separate axes because profit and demand have different units.
ax3 = axes[1, 0]
scenario_names = [item["scenario"] for item in scenario_results]
scenario_profits = [item["profit"] for item in scenario_results]
scenario_demands = [item["demand"] for item in scenario_results]
positions = np.arange(len(scenario_names))
bar_width = 0.35

profit_bars = ax3.bar(
    positions - bar_width / 2,
    scenario_profits,
    bar_width,
    color="lightblue",
    label="Profit (USD)",
)
ax3_twin = ax3.twinx()
demand_bars = ax3_twin.bar(
    positions + bar_width / 2,
    scenario_demands,
    bar_width,
    color="lightcoral",
    label="Demand (units)",
)
ax3.set_xlabel("Seasonal scenario")
ax3.set_ylabel("Profit (USD)")
ax3_twin.set_ylabel("Demand (units)")
ax3.set_title("Optimized Results by Seasonal Scenario")
ax3.set_xticks(positions)
ax3.set_xticklabels(scenario_names, rotation=30, ha="right")
ax3.legend(
    handles=[profit_bars, demand_bars],
    labels=["Profit (USD)", "Demand (units)"],
    loc="upper left",
)
ax3.grid(True, axis="y", alpha=0.3)

# This panel reports calculated quantities without implying unsupported market effects.
ax4 = axes[1, 1]
ax4.axis("off")
summary_text = (
    "Optimized strategy\n"
    f"  Price: ${results['price']:.2f} per unit\n"
    f"  Marketing spend: ${results['marketing_spend']:,.2f}\n"
    f"  Partnership intensity: {results['partnership_level']:.3f}\n"
    f"  Order quantity: {results['order_quantity']:.1f} units\n"
    "\nModeled outcomes\n"
    f"  Demand: {results['demand']:.1f} units\n"
    f"  Sales: {results['sales']:.1f} units\n"
    f"  Leftovers: {results['leftovers']:.1f} units\n"
    f"  Profit: ${results['profit']:,.2f}\n"
    "\nReference comparison\n"
    f"  Profit difference: {improvement:+.2f}%\n"
    "\nInterpretation\n"
    "  Results depend on the stated demand and cost assumptions.\n"
    "  The model is single-period and deterministic."
)
ax4.text(
    0.04,
    0.96,
    summary_text,
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="top",
    fontfamily="monospace",
    bbox={"boxstyle": "round", "facecolor": "lightgray", "alpha": 0.8},
)

fig.tight_layout()
plt.show()
print("Demand-shaping analysis visualization complete")

## Summary

This notebook combines demand shaping with a single-period order decision. Price, marketing spend, and partnership intensity affect the deterministic demand estimate; price, order quantity, and leftover inventory then determine revenue and costs in the profit objective. Differential evolution searches the specified continuous bounds for a high-profit decision vector, and the seasonal scenarios show how that same model responds to changes in its seasonal input.

The reported comparison is relative to the stated reference policy, not evidence of a guaranteed improvement in a real market. The optimizer's result depends on the demand response, cost assumptions, bounds, and numerical search. Demand uncertainty, customer segments, long-term market positioning, and competitive responses are outside this model's scope.